In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore

In [3]:
loader = PyPDFLoader("../data/medical_report.pdf")
docs = loader.load()

In [5]:
split = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
splitted_docs = split.split_documents(docs)

In [6]:
embedded = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")
store_db = InMemoryVectorStore.from_documents(
    documents=splitted_docs,
    embedding=embedded
)

In [9]:
from langchain_groq import ChatGroq
from langgraph.graph import START,END, StateGraph
from pydantic import BaseModel,Field

In [10]:
class ragState(BaseModel):
    question: str = Field(description="User Question")
    documents : list = []
    context : str = Field(description="Context Data for user question", default="")
    answer : str = Field(description="Final Answer...", default="")
    

In [13]:
def retrive(state:ragState)-> ragState:
    docs = store_db.similarity_search(query=state.question)
    state.documents=docs
    return state

In [14]:
def context(state:ragState)-> ragState:
    context=""
    for doc in state.documents:
        context+= doc.page_content
    state.context = context
    return state


In [15]:
llm = ChatGroq(model="openai/gpt-oss-120b")

In [16]:

def generate_node(state:ragState) -> ragState:
    prompt = f"""
        You are a assistant and provide the answer for user question based on the provided context.
        If you don't find the relevant answer, then just say 'I dont know.'.
        Context is: {state.context},
        Questions is: {state.question}
    """
    res = llm.invoke(prompt)
    state.answer = res.content
    return state

In [20]:
graph = StateGraph(ragState)

graph.add_node("retrieve_node", retrive)
graph.add_node("create_context_node", context)
graph.add_node("generate_node", generate_node)

graph.add_edge(START, "retrieve_node")
graph.add_edge("retrieve_node", "create_context_node")
graph.add_edge("create_context_node", "generate_node")
graph.add_edge("generate_node", END)

graph = graph.compile()

In [22]:
res = graph.invoke({"question":"What is her age"})
print(res["answer"])

Her age is **27 years**.
